### Topic 4: Many-to-One Relationship Concept
**Explanation:**
In a many-to-one relationship (e.g., a blog where one user can create many posts), the foreign key is **not unique** in the child table. The child table (`posts`) has its own primary key (`id`). The foreign key column (`user_id`) references the parent table (`users`) and can have duplicate values.

In [12]:
import mysql.connector

#Modify the connection to include the database name
database_name="Code_with_tim_test_database"
db=mysql.connector.connect(
    host="localhost",
    user="root",
    password="gpt67rSy",
    database=database_name, # <-- We are now connecting to this database
    buffered=True,# best practice - for parallel multiple queries,prevents error,standard practice
)
if db.is_connected():
    print(f"Successfully connected to the '{db.database}' database.")

#We can now create cursor to work within this database
my_cursor=db.cursor()

# You can now execute queries that affect this database,
# like creating tables, inserting data, etc.

Successfully connected to the 'code_with_tim_test_database' database.


### 1. Create the Parent Table (userss)

In [13]:
create_userss="""
CREATE TABLE IF NOT EXISTS userss(
    id INT AUTO_INCREMENT NOT NULL UNIQUE,
    username VARCHAR(50) NOT NULL UNIQUE,
    email VARCHAR(100) NOT NULL UNIQUE,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY(id)
)
"""
my_cursor.execute(create_userss)

###  Child table: posts (with foreign key)

In [14]:
create_posts="""
CREATE TABLE IF NOT EXISTS posts(
    id INT AUTO_INCREMENT NOT NULL UNIQUE,
    title VARCHAR(200) NOT NULL,
    content TEXT,
    user_id INT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY(id),
    FOREIGN KEY (user_id) REFERENCES userss(id) ON DELETE CASCADE

)
"""

my_cursor.execute(create_posts)

### STEP 2: Insert Users (MUST commit)

In [15]:
insert_user="INSERT INTO userss(username,email) VALUES (%s,%s)"

userss_data=[
    ("Kristal","kristal@gmail.com"),
    ("Biraj","Biraj@gmail.com"),
    ("Tim","Tim@gmail.com")
]

In [16]:
user_ids=[] #store IDs of inserted users
for user in userss_data:
    my_cursor.execute(insert_user,user)
    user_ids.append(my_cursor.lastrowid)
    print(f"Inserted user: {user[0]} with ID: {my_cursor.lastrowid}")


db.commit() # IMPORTANT: Commit after INSERT

Inserted user: Kristal with ID: 1
Inserted user: Biraj with ID: 2
Inserted user: Tim with ID: 3


#### STEP 3: Insert Posts (MUST commit)
Each user will have multiple posts (one-to-many relationship)

In [ ]:
# my_cursor.execute("DROP TABLE IF EXISTS post")

In [22]:
insert_post="""
INSERT INTO posts (title, content, user_id) VALUES (%s, %s, %s)
"""

In [23]:
posts_data = [
    # kristal's posts (user_id = 1)
    ("MySQL Basics", "Learning about foreign keys...", user_ids[0]),
    ("Python Tips", "10 tips for beginners...", user_ids[0]),
    ("Database Design", "How to structure tables...", user_ids[0]),
    
    # Biraj's posts (user_id = 2)
    ("Web Development", "HTML/CSS fundamentals...", user_ids[1]),
    ("JavaScript Tricks", "Closures and scope...", user_ids[1]),
    
    # Tim's posts (user_id = 3)
    ("Machine Learning", "Introduction to neural networks...", user_ids[2]),
    ("Data Science", "Pandas tutorial...", user_ids[2]),
    ("AI Ethics", "Important considerations...", user_ids[2]),
    ("Python Libraries", "NumPy vs Pandas...", user_ids[2])
]

In [ ]:
for post in posts_data:
    my_cursor.execute(insert_post, post)
    print(f"Inserted post: '{post[0]}' for user_id: {post[2]}")

db.commit() # important commit after insert
print(f"\n{len(posts_data)} posts inserted and committed!\n")

Inserted post: 'MySQL Basics' for user_id: 1
Inserted post: 'Python Tips' for user_id: 1
Inserted post: 'Database Design' for user_id: 1
Inserted post: 'Web Development' for user_id: 2
Inserted post: 'JavaScript Tricks' for user_id: 2
Inserted post: 'Machine Learning' for user_id: 3
Inserted post: 'Data Science' for user_id: 3
Inserted post: 'AI Ethics' for user_id: 3
Inserted post: 'Python Libraries' for user_id: 3

9 posts inserted and committed!



### STEP 4: Query Data - JOIN to get user info with posts

In [27]:
print("=" * 50)
print("ALL POSTS WITH USER INFORMATION:")
print("=" * 50)

query = """
SELECT posts.id, posts.title, posts.content, userss.username, userss.email
FROM posts
JOIN userss ON posts.user_id = userss.id
ORDER BY userss.username, posts.created_at
"""

my_cursor.execute(query)
results = my_cursor.fetchall()

for post in results:
    print(f"\n📝 Post ID: {post[0]}")
    print(f"   Title: {post[1]}")
    print(f"   Content: {post[2][:50]}..." if len(post[2]) > 50 else f"   Content: {post[2]}")
    print(f"   Author: {post[3]} ({post[4]})")

ALL POSTS WITH USER INFORMATION:

📝 Post ID: 4
   Title: Web Development
   Content: HTML/CSS fundamentals...
   Author: Biraj (Biraj@gmail.com)

📝 Post ID: 5
   Title: JavaScript Tricks
   Content: Closures and scope...
   Author: Biraj (Biraj@gmail.com)

📝 Post ID: 13
   Title: Web Development
   Content: HTML/CSS fundamentals...
   Author: Biraj (Biraj@gmail.com)

📝 Post ID: 14
   Title: JavaScript Tricks
   Content: Closures and scope...
   Author: Biraj (Biraj@gmail.com)

📝 Post ID: 1
   Title: MySQL Basics
   Content: Learning about foreign keys...
   Author: Kristal (kristal@gmail.com)

📝 Post ID: 2
   Title: Python Tips
   Content: 10 tips for beginners...
   Author: Kristal (kristal@gmail.com)

📝 Post ID: 3
   Title: Database Design
   Content: How to structure tables...
   Author: Kristal (kristal@gmail.com)

📝 Post ID: 10
   Title: MySQL Basics
   Content: Learning about foreign keys...
   Author: Kristal (kristal@gmail.com)

📝 Post ID: 11
   Title: Python Tips
   Content: 1

### STEP 5: Get All Posts for a Specific User

In [30]:
print("\n" + "=" * 50)
print("POSTS BY SPECIFIC USER:")
print("=" * 50)

# Get posts for Tim (user_id = user_ids[0])
my_cursor.execute("SELECT title, content FROM posts WHERE user_id = %s", (user_ids[0],))
kristal_posts = my_cursor.fetchall()

print(f"\n📚 Posts by Kristal (ID: {user_ids[0]}):")
for i, post in enumerate(kristal_posts, 1):
    print(f"   {i}. {post[0]}")


POSTS BY SPECIFIC USER:

📚 Posts by Kristal (ID: 1):
   1. MySQL Basics
   2. Python Tips
   3. Database Design
   4. MySQL Basics
   5. Python Tips
   6. Database Design


###  STEP 6: Count Posts Per User

In [32]:
print("\n" + "=" * 50)
print("POST COUNT PER USER:")
print("=" * 50)

my_cursor.execute("""
    SELECT userss.username, COUNT(posts.id) as post_count
    FROM userss
    LEFT JOIN posts ON userss.id = posts.user_id
    GROUP BY userss.id, userss.username
""")

counts = my_cursor.fetchall()
for username, count in counts:
    print(f"   {username}: {count} post(s)")


POST COUNT PER USER:
   Biraj: 4 post(s)
   Kristal: 6 post(s)
   Tim: 8 post(s)


### STEP 7: Clean Up (Optional - DELETE)

In [39]:
print("\n" + "=" * 50)
print("CASCADE DELETE DEMONSTRATION:")
print("=" * 50)

# Show posts before deletion
my_cursor.execute("SELECT COUNT(*) FROM posts WHERE user_id = %s", (user_ids[0],))
print(f"Kristal had {my_cursor.fetchone()[0]} posts")

# Delete a user - posts will be automatically deleted due to ON DELETE CASCADE
my_cursor.execute("DELETE FROM userss WHERE id = %s", (user_ids[0],))
db.commit()  # IMPORTANT: Commit after DELETE

# Check posts for that user
my_cursor.execute("SELECT COUNT(*) FROM posts WHERE user_id = %s", (user_ids[0],))
print(f"After deleting Kristal, he now has {my_cursor.fetchone()[0]} posts (all deleted with CASCADE)")


CASCADE DELETE DEMONSTRATION:
Kristal had 0 posts
After deleting Kristal, he now has 0 posts (all deleted with CASCADE)


# Close connection

In [40]:
my_cursor.close()
db.close()
print("\n✅ Connection closed!")


✅ Connection closed!
